# Exercises XP
## Introduction to Data Analysis

This notebook covers Exercises 1 to 10 in English.
Tools: Python, Pandas, Matplotlib, Seaborn, Jupyter.

In [ ]:
# Common imports used throughout the notebook
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')

# Base directory (the folder where this notebook lives)
BASE_DIR = os.getcwd()
print('Working directory:', BASE_DIR)

## Exercise 1: Introduction to Data Analysis

### 1) What is data analysis?
Data analysis is the process of **collecting, cleaning, exploring, transforming and interpreting** data in order to extract meaningful insights. It mixes statistics, programming and domain knowledge to turn raw data into knowledge that supports decisions, predictions and storytelling.

### 2) Why is data analysis important in modern contexts?
In the digital era, data is produced continuously by applications, sensors, social networks and transactions. Analyzing this data allows organizations to:
- make **evidence-based decisions** instead of relying on intuition,
- **detect trends, patterns and anomalies** early,
- **optimize processes** (costs, delays, quality, supply chain),
- **personalize** offers, products and user experiences,
- **predict** future events (demand, churn, risk).

### 3) Three application areas
1. **Healthcare** — analysis of patient records, medical imaging and biological markers to improve diagnosis, monitor diseases and evaluate treatments.
2. **Finance & Marketing** — customer segmentation, credit scoring, fraud detection, A/B testing, and campaign performance tracking.
3. **Industry & Logistics** — predictive maintenance from IoT sensors, route optimization, inventory management and reduction of downtime.

## Exercise 2 : Dataset Loading and Initial Analysis

**Objective:** Load three datasets, display the first rows and provide a short description.

Datasets used:
- **How Much Sleep Do Americans Really Get?** → `Time Americans Spend Sleeping.csv`
- **Global Trends in Mental Health Disorder** → `Mental health Depression disorder Data.csv`
- **Credit Card Approvals** → `clean_dataset.csv` (cleaned version of `crx.csv`)

In [ ]:
# 2.1 — Sleep dataset
sleep_df = pd.read_csv(os.path.join(BASE_DIR, 'Time Americans Spend Sleeping.csv'))
print('Shape:', sleep_df.shape)
sleep_df.head()

**Description — Sleep dataset:** Yearly average hours of sleep per day for Americans (2003 onwards), broken down by **age group**, **sex** and **type of day** (weekday / weekend / all days). It is useful for studying sleep trends over time and group comparisons.

In [ ]:
# 2.2 — Mental health dataset
mental_df = pd.read_csv(os.path.join(BASE_DIR, 'Mental health Depression disorder Data.csv'))
print('Shape:', mental_df.shape)
mental_df.head()

**Description — Mental health dataset:** Global prevalence (in %) of several mental health disorders — schizophrenia, bipolar disorder, eating disorders, anxiety, drug use, depression and alcohol use — by **country** (`Entity`/`Code`) and by **year**. Useful for international and temporal comparisons.

In [ ]:
# 2.3 — Credit Card Approvals dataset
credit_df = pd.read_csv(os.path.join(BASE_DIR, 'clean_dataset.csv'))
print('Shape:', credit_df.shape)
credit_df.head()

**Description — Credit Card Approvals dataset:** Anonymized applicant features (gender, age, debt, marital status, industry, ethnicity, years employed, credit score, income, etc.) and the final **`Approved`** decision (0 = refused, 1 = approved). Typical use case: binary classification.

## Exercise 3 : Identifying Data Types (Qualitative vs Quantitative)

**Reminder**
- **Quantitative**: numerical values that can be measured or counted (continuous or discrete).
- **Qualitative**: categories or labels (nominal or ordinal).

In [ ]:
def classify_columns(df, name):
    """Return a small DataFrame summarizing dtype and qualitative/quantitative nature."""
    rows = []
    for col in df.columns:
        kind = 'quantitative' if pd.api.types.is_numeric_dtype(df[col]) else 'qualitative'
        rows.append({'column': col, 'dtype': str(df[col].dtype), 'type': kind})
    print(f'--- {name} ---')
    return pd.DataFrame(rows)

classify_columns(sleep_df, 'Sleep dataset')

In [ ]:
classify_columns(mental_df, 'Mental health dataset')

In [ ]:
classify_columns(credit_df, 'Credit Card Approvals dataset')

### Justifications

**Sleep dataset**
- `Year`, `Avg hrs per day sleeping`, `Standard Error` → **quantitative** (numerical measurements).
- `Period`, `Type of Days`, `Age Group`, `Activity`, `Sex` → **qualitative** (categorical labels).

**Mental health dataset**
- `Year` and every disorder percentage column (`Schizophrenia (%)`, `Depression (%)`, ...) → **quantitative**.
- `Entity` (country name) and `Code` (ISO country code) → **qualitative**.

**Credit Card Approvals dataset**
- `Age`, `Debt`, `YearsEmployed`, `CreditScore`, `Income` → **quantitative** (real measurements).
- `Industry`, `Ethnicity`, `Citizen`, `ZipCode` → **qualitative** (categories/labels).
- `Gender`, `Married`, `BankCustomer`, `PriorDefault`, `Employed`, `DriversLicense`, `Approved` → **qualitative encoded as 0/1** (binary categories rather than measurements).

## Exercise 4 : Exploring Data Types with the Iris dataset

In [ ]:
# Load the Iris dataset from the local CSV file (originally from Kaggle)
iris_df = pd.read_csv(os.path.join(BASE_DIR, 'Iris.csv'))
print('Shape:', iris_df.shape)
iris_df.head()

In [ ]:
classify_columns(iris_df, 'Iris dataset')

### Iris — column-by-column classification
- `Id` → **quantitative** in dtype, but it is just a row identifier (no analytical meaning).
- `SepalLengthCm`, `SepalWidthCm`, `PetalLengthCm`, `PetalWidthCm` → **quantitative** — physical measurements in centimeters.
- `Species` → **qualitative** — three categories: *Iris-setosa*, *Iris-versicolor*, *Iris-virginica*.

Statistical calculations and basic visualizations on the quantitative columns:

In [ ]:
# Mean, median and mode for the numerical features
numeric_cols = ['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']
stats = pd.DataFrame({
    'mean': iris_df[numeric_cols].mean(),
    'median': iris_df[numeric_cols].median(),
    'mode': iris_df[numeric_cols].mode().iloc[0]
})
stats

In [ ]:
# Histograms of the four numerical features
iris_df[numeric_cols].hist(bins=20, figsize=(10, 6), color='steelblue', edgecolor='black')
plt.suptitle('Iris — Distribution of numerical features')
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart showing how many samples belong to each species
species_counts = iris_df['Species'].value_counts()
species_counts.plot(kind='bar', color=['#4c72b0', '#dd8452', '#55a868'], edgecolor='black')
plt.title('Iris — Number of samples per species')
plt.xlabel('Species')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

## Exercise 5 : Observation Skills on the Sleep dataset

In [ ]:
# General info about the dataset
sleep_df.info()
print('\nUnique values per categorical column:')
for col in ['Period', 'Type of Days', 'Age Group', 'Activity', 'Sex']:
    print(f'- {col}: {sleep_df[col].unique()}')

### Columns of interest and why

- **`Year`** + **`Avg hrs per day sleeping`** → ideal for **trend analysis**: are Americans sleeping more or less over the years?
- **`Sex`** + **`Avg hrs per day sleeping`** → useful for **group comparison** between men and women.
- **`Age Group`** + **`Avg hrs per day sleeping`** → comparison across age groups (15–24, 25–34, …, 65+).
- **`Type of Days`** (weekdays vs weekends vs all days) → behavioral comparison between work week and weekend.
- **`Standard Error`** → quality of the measurement, can be used to add error bars on plots.

In [ ]:
# Quick trend visualization: average sleep hours by year (Both sexes, all days)
trend = sleep_df[(sleep_df['Sex'] == 'Both') & (sleep_df['Type of Days'] == 'All days')]
trend = trend.groupby('Year')['Avg hrs per day sleeping'].mean().reset_index()

plt.figure(figsize=(9, 4))
sns.lineplot(data=trend, x='Year', y='Avg hrs per day sleeping', marker='o')
plt.title('Average sleep hours per day in the US — All days, both sexes')
plt.ylabel('Hours of sleep')
plt.show()

## Exercise 6 : Structured or Unstructured Data?

| Data source | Type | Reason |
|---|---|---|
| Company financial reports in Excel | **Structured** | Stored in rows/columns with a defined schema; directly queryable. |
| Photographs uploaded to a social media platform | **Unstructured** | Binary image data; no inherent tabular schema. |
| Collection of news articles on a website | **Unstructured** | Free-form text with HTML formatting; needs parsing/NLP. |
| Inventory data in a relational database | **Structured** | Relational schema with tables, columns, keys and constraints. |
| Recorded interviews from a market research study | **Unstructured** | Raw audio (and/or transcripts); requires transcription and NLP to extract structure. |

## Exercise 7 : Transforming Unstructured Data into Structured Data

**1) Series of travel blog posts**  
Apply **NLP**: tokenize text, run **Named Entity Recognition** to extract locations, dates and people, and use **sentiment analysis** to score each post. Build a DataFrame with columns such as `post_id`, `country`, `city`, `date`, `theme`, `sentiment`, `length`.

**2) Audio recordings of customer service calls**  
Use **Automatic Speech Recognition (ASR)** (e.g. Whisper) to transcribe the audio, then apply NLP to extract intent, topic, sentiment and resolution status. Final structured columns: `call_id`, `date`, `duration`, `agent_id`, `topic`, `sentiment`, `resolved`.

**3) Handwritten notes from a brainstorming session**  
Run **OCR** (Optical Character Recognition, e.g. Tesseract) to digitize the text, then segment it into ideas and tag them with keywords/priority. Final columns: `note_id`, `idea`, `keywords`, `priority`, `author`.

**4) Cooking video tutorial**  
Extract the **subtitles** if available, otherwise use ASR. Then perform **temporal segmentation** to detect cooking steps and extract ingredients and durations. Final columns: `step_id`, `timestamp_start`, `timestamp_end`, `action`, `ingredient`, `quantity`, `tool`.

## Exercise 8 : Import the Titanic `train.csv` Dataset

The Titanic `train.csv` file is available locally in this folder (originally sourced from Kaggle / mirrored on GitHub).

In [ ]:
# Load the Titanic training dataset
train_df = pd.read_csv(os.path.join(BASE_DIR, 'train.csv'))
print('Shape:', train_df.shape)
train_df.head()

In [ ]:
# Quick overview of the columns and missing values
train_df.info()
print('\nMissing values per column:')
print(train_df.isna().sum())

## Exercise 9 : Export a DataFrame to Excel and JSON

In [ ]:
# Build a simple DataFrame
df_simple = pd.DataFrame({
    'name':  ['Alice', 'Bob', 'Charlie', 'Diana'],
    'age':   [25, 30, 35, 28],
    'score': [88.5, 91.2, 79.9, 95.4],
    'city':  ['Paris', 'Berlin', 'New York', 'Tokyo']
})
df_simple

In [ ]:
# Export to Excel and JSON
excel_path = os.path.join(BASE_DIR, 'df_simple.xlsx')
json_path  = os.path.join(BASE_DIR, 'df_simple.json')

# openpyxl is required for the .xlsx engine
df_simple.to_excel(excel_path, index=False)
df_simple.to_json(json_path, orient='records', indent=2)

print('Excel file written ->', excel_path)
print('JSON file written  ->', json_path)

## Exercise 10 : Reading JSON Data

We read the sample JSON dataset provided in this folder (`posts.json`, equivalent to https://jsonplaceholder.typicode.com/posts).

In [ ]:
# Read the JSON file with pandas
posts_df = pd.read_json(os.path.join(BASE_DIR, 'posts.json'))
print('Shape:', posts_df.shape)
posts_df.head()

In [ ]:
# Same operation but reading directly from a remote URL
JSON_URL = 'https://jsonplaceholder.typicode.com/posts'
try:
    posts_remote = pd.read_json(JSON_URL)
    print('Shape:', posts_remote.shape)
    display(posts_remote.head())
except Exception as e:
    print('Could not fetch the remote JSON (offline?):', e)


# Exercises XP Gold
## Data analysis, understanding and preprocessing

All exercises below are written in English with English comments.
They use the CSV files already available in this folder.

## Exercise 1: Basic Data Analysis with Kaggle

We pick a simple Kaggle dataset (the **Walmart Sales** retail dataset, file `sales data-set.csv`) and compute the **mean, median and standard deviation** of the `Weekly_Sales` column.
*(The instructions suggest Boston Housing as an example, but that dataset is not present in this folder — any simple dataset works.)*

In [ ]:
# Load a simple Kaggle dataset and compute mean / median / std on one column
sales_df = pd.read_csv(os.path.join(BASE_DIR, 'sales data-set.csv'))
print('Shape:', sales_df.shape)
sales_df.head()

In [ ]:
col = 'Weekly_Sales'
print(f'Statistics for {col}')
print(f'  mean   = {sales_df[col].mean():.2f}')
print(f'  median = {sales_df[col].median():.2f}')
print(f'  std    = {sales_df[col].std():.2f}')

## Exercise 2: Comparative Analysis of Retail Data

- **Structured data:** Walmart Sales dataset → `sales data-set.csv` (+ `stores data-set.csv`).
- **Unstructured data:** Women's Clothing E-Commerce Reviews → `Womens Clothing E-Commerce Reviews.csv` (column `Review Text`).

In [ ]:
# --- Structured retail data: sales trends, customer purchase patterns, store performance ---
stores_df = pd.read_csv(os.path.join(BASE_DIR, 'stores data-set.csv'))

# Convert dates and merge with the store metadata
sales_df['Date'] = pd.to_datetime(sales_df['Date'], dayfirst=True)
retail = sales_df.merge(stores_df, on='Store', how='left')

# Monthly sales trend
monthly = retail.groupby(retail['Date'].dt.to_period('M'))['Weekly_Sales'].sum()

plt.figure(figsize=(10, 4))
monthly.plot()
plt.title('Total weekly sales aggregated by month')
plt.ylabel('Sales ($)')
plt.show()

In [ ]:
# Top 10 stores by total sales
top_stores = retail.groupby('Store')['Weekly_Sales'].sum().sort_values(ascending=False).head(10)
top_stores.plot(kind='bar', color='steelblue', edgecolor='black', figsize=(9, 4))
plt.title('Top 10 stores by total sales')
plt.ylabel('Total sales ($)')
plt.show()

# Performance by store type
print('\nAverage weekly sales per store type:')
print(retail.groupby('Type')['Weekly_Sales'].mean().round(2))

In [ ]:
# --- Unstructured data: customer reviews ---
reviews_df = pd.read_csv(os.path.join(BASE_DIR, 'Womens Clothing E-Commerce Reviews.csv'), index_col=0)
print('Shape:', reviews_df.shape)
reviews_df.head(3)

In [ ]:
# Simple proxy for sentiment: the explicit Rating column (1-5)
print('Rating distribution:')
print(reviews_df['Rating'].value_counts().sort_index())

reviews_df['Rating'].value_counts().sort_index().plot(
    kind='bar', color='salmon', edgecolor='black', figsize=(7, 3))
plt.title('Distribution of customer ratings')
plt.xlabel('Rating (1 = worst, 5 = best)')
plt.ylabel('Number of reviews')
plt.show()

# Recommendation rate
rec_rate = reviews_df['Recommended IND'].mean() * 100
print(f'Recommendation rate: {rec_rate:.1f}% of customers recommend the product.')

In [ ]:
# Most frequent words in the review text (very simple text mining)
from collections import Counter
import re

STOPWORDS = set('''the a an and or but if to of in on for with is are was were be been being
this that these those it its as at by from i you he she they we my your our their not no so do does did just
have has had can could would should will i\'m it\'s don\'t didn\'t '''.split())

text = ' '.join(reviews_df['Review Text'].dropna().astype(str).str.lower().tolist())
words = [w for w in re.findall(r'[a-z]+', text) if w not in STOPWORDS and len(w) > 2]
common = Counter(words).most_common(15)
print('Top 15 most frequent words in reviews:')
for w, c in common:
    print(f'  {w:15s} {c}')

### Comparison and challenges

**Structured (sales) insights** are direct and quantitative:
- clear monthly/seasonal sales trends,
- ranked store performance,
- difference between store types A/B/C,
- impact of holidays via the `IsHoliday` flag.

**Unstructured (reviews) insights** require text mining but reveal *why* customers behave the way they do:
- predominant sentiment (mostly 4-5 star ratings → positive overall),
- frequently mentioned topics (fit, size, fabric, color, dress, …),
- overall satisfaction estimated from recommendation rate.

**Challenges with unstructured data**
- Missing values in free-text fields (`Review Text` and `Title` had nulls).
- Inconsistent vocabulary, spelling mistakes, abbreviations.
- Heavy preprocessing (lower-casing, stopword removal, lemmatization, …) before any analysis.
- Sentiment and topics are *interpretations*, not direct measurements — they depend on the NLP method chosen.
- Computationally more expensive than tabular aggregations.

## Exercise 3: Basic Data Exploration in E-Commerce

Dataset: `data.csv` (the classic UK Online Retail dataset).

In [ ]:
# Load the E-Commerce dataset (the file is encoded as latin-1)
ecom_df = pd.read_csv(os.path.join(BASE_DIR, 'data.csv'), encoding='latin-1')
print('Shape:', ecom_df.shape)
ecom_df.head()

In [ ]:
# Basic information about the dataset
print('Number of rows   :', ecom_df.shape[0])
print('Number of columns:', ecom_df.shape[1])
print('Column names     :', list(ecom_df.columns))
ecom_df.info()

### Structured columns in the E-Commerce dataset
- `InvoiceNo` → identifier (text/code) — categorical.
- `StockCode` → product code — categorical/identifier.
- `Quantity` → integer measurement — numerical.
- `InvoiceDate` → date/time — structured timestamp.
- `UnitPrice` → decimal measurement — numerical.
- `CustomerID` → identifier — categorical/numerical id.
- `Country` → fixed list of countries — categorical.

The only **semi-unstructured** field is `Description` (free-text product description).

### What unstructured data could complement this dataset?
- **Customer reviews** of each product (free text + star rating).
- **Product images** uploaded with each listing.
- **Customer support emails / chat transcripts**.
- **Social media posts** mentioning the brand or products.

### How could the unstructured data be used?
- Combine sales numbers with **sentiment from reviews** to find products that sell well *and* are loved by customers.
- Use **NLP on emails** to detect recurring problems (returns, delivery issues) tied to specific products.
- Run **image analysis** to enrich the catalog with auto-generated tags (color, style, category) used to build recommendation engines.
- Use **social listening** to forecast demand spikes before they show up in sales.

## Exercise 4: Public Transportation Dataset — Metro Interstate Traffic Volume

In [ ]:
traffic_df = pd.read_csv(os.path.join(BASE_DIR, 'Metro_Interstate_Traffic_Volume.csv'))
print('Shape:', traffic_df.shape)
traffic_df.head()

In [ ]:
traffic_df.info()
print('\nSample values per column:')
for col in traffic_df.columns:
    print(f'- {col}: {traffic_df[col].iloc[0]!r}')

### Structured vs unstructured elements

| Column | Type | Reason |
|---|---|---|
| `holiday` | **Structured (categorical)** | Fixed list of holiday names + `None`. |
| `temp` | **Structured (numerical)** | Temperature in Kelvin, a measurement. |
| `rain_1h`, `snow_1h` | **Structured (numerical)** | Continuous measurements (mm). |
| `clouds_all` | **Structured (numerical)** | Percentage of cloud cover. |
| `weather_main` | **Structured (categorical)** | Short fixed-vocabulary label (Clouds, Clear, Rain, …). |
| `weather_description` | **Semi-unstructured** | Short free-form text ("scattered clouds", "light rain", …). It has a small vocabulary but is not strictly enumerated. |
| `date_time` | **Structured (timestamp)** | ISO date and time. |
| `traffic_volume` | **Structured (numerical)** | Integer count of vehicles. |

Overall the dataset is highly structured. The closest thing to unstructured data is `weather_description`, which would benefit from being mapped to a controlled vocabulary or to numerical severity scores before modelling.

## Exercise 5: Basic Data Analysis on a Movie Ratings Dataset (MovieLens)

In [ ]:
# Load MovieLens ratings file
ratings_df = pd.read_csv(os.path.join(BASE_DIR, 'rating.csv'))
print('Shape:', ratings_df.shape)
ratings_df.head()

In [ ]:
ratings_df.info()
print('\nQuick statistics on ratings:')
print(ratings_df['rating'].describe())

### Structured elements in `rating.csv`
- `userId` → integer user identifier.
- `movieId` → integer movie identifier.
- `rating` → numerical (0.5 to 5.0, step 0.5).
- `timestamp` → date/time when the rating was given.

### Why is this dataset considered structured?
- Every row follows the **same schema** (4 fixed columns).
- Each column has a **well-defined type and domain** (id, id, number, datetime).
- The file can be loaded directly into a relational table or a DataFrame without any parsing/extraction step.
- Aggregations (group by user, by movie, by time) are immediate.

## Exercise 6: Creating a Synthetic Product Catalog with Faker

We generate **500 synthetic products** for an e-commerce platform using the `Faker` library.

In [ ]:
# Make sure Faker is installed; the line below installs it from the notebook if missing.
try:
    from faker import Faker
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'Faker'])
    from faker import Faker

print('Faker is ready.')

In [ ]:
import uuid

fake = Faker()
Faker.seed(42)  # reproducible catalog

N_PRODUCTS = 500
products = []
for _ in range(N_PRODUCTS):
    products.append({
        'product_id'  : str(uuid.uuid4()),
        'name'        : fake.catch_phrase(),
        'description' : fake.text(max_nb_chars=160),
        'price'       : round(fake.pyfloat(min_value=5, max_value=500, right_digits=2), 2),
    })

catalog_df = pd.DataFrame(products)
print('Catalog shape:', catalog_df.shape)
catalog_df.head()

In [ ]:
# Save the synthetic catalog to disk for reuse
catalog_path = os.path.join(BASE_DIR, 'synthetic_catalog.csv')
catalog_df.to_csv(catalog_path, index=False)
print('Synthetic catalog saved to:', catalog_path)

# Quick descriptive statistics on the price column
print('\nPrice statistics:')
print(catalog_df['price'].describe().round(2))